# Waymaker × Gemma 4 — LoRA/QLoRA fine-tuning experiment (Google Colab)

**Experimental only. Not wired into the live Waymaker backend. Not production-ready.**

This notebook fine-tunes a Gemma 4 instruction model with QLoRA for **behavior and
output formatting only**:

- answer **only** from the `evidence_pack` in the user message
- refuse / defer when evidence is insufficient
- never invent fees, periods, document lists, or eligibility rules
- produce a consistent 6-section Korean guidance structure
- preserve source grounding (cite the sources used)

It must **not** teach the model visa rules, fees, periods, eligibility requirements,
or document lists. The bundled sample data uses clearly-marked fake placeholder
excerpts — no real legal facts.

## Two-stage strategy

| Stage | Model | Purpose |
|---|---|---|
| 1 (default) | `google/gemma-4-E4B-it` | **Smoke test only** — verify dataset format, chat template, LoRA/QLoRA code, adapter save/load, inference, eval script |
| 2 | `google/gemma-4-12B-it` | **Actual quality experiment** — Waymaker-style answer quality, evidence grounding, hallucination/refusal behavior |

Run Stage 1 end-to-end first. Only after everything works, flip the model in the
clearly marked cell below and rerun.

**Runtime:** GPU runtime required (Runtime → Change runtime type → GPU).
T4 is fine for E4B smoke tests; prefer Colab Pro **L4 or A100** for the 12B model.

Training is on **final visible assistant answers only** — no hidden chain-of-thought
is included in the data or the loss.

In [ ]:
# 1) Install dependencies
# Pinned loosely; if model loading fails with an architecture error, upgrade transformers.
%pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub

In [ ]:
# 2) Hugging Face authentication
# Preferred: Colab secret named HF_TOKEN (key icon in the left sidebar).
# Fallback: environment variable HF_TOKEN.
# You must have accepted the Gemma license on the model pages on huggingface.co.
import os

hf_token = None
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass
hf_token = hf_token or os.environ.get("HF_TOKEN")
assert hf_token, "Set a Colab secret or env var named HF_TOKEN (with access to the Gemma models)."

from huggingface_hub import login
login(token=hf_token)
print("Hugging Face login OK")

In [ ]:
# 3) Experiment configuration (conservative smoke-test defaults)

# ----- Stage 1 (default): smoke test model -----
MODEL_ID = "google/gemma-4-E4B-it"

# Conservative defaults for the first smoke test. See OOM notes at the bottom.
MAX_SEQ_LENGTH = 1024            # OOM? reduce to 768 or 512 first
PER_DEVICE_TRAIN_BATCH_SIZE = 1  # keep at 1; raise grad accum instead
GRADIENT_ACCUMULATION_STEPS = 8  # 4 or 8; reduce if a single step is too slow / OOMs
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
USE_QLORA = True                 # 4-bit base model + LoRA adapters (recommended on Colab)

OUTPUT_DIR = "/content/waymaker-gemma4-lora"
RUN_NAME = MODEL_ID.split("/")[-1] + "-waymaker-lora"
print("Config:", MODEL_ID, "| QLoRA:", USE_QLORA)

## ⚠️ STAGE 2 SWITCH — run this cell ONLY after the E4B smoke test succeeds

Uncomment the line below and **rerun every cell from here down** (model load → train →
save → inference). E4B results are *not* a quality signal — only 12B results count
for the actual experiment. Use an L4 or A100 runtime for 12B if possible.

In [ ]:
# ===== STAGE 2: switch to the quality-experiment model =====
# Uncomment AFTER the E4B smoke test passes end-to-end:

# MODEL_ID = "google/gemma-4-12B-it"

RUN_NAME = MODEL_ID.split("/")[-1] + "-waymaker-lora"
print("Active model:", MODEL_ID)

In [ ]:
# 4) Load the dataset (messages-format JSONL)
# Option A (default): upload train.sample.jsonl / eval.sample.jsonl from
#   experiments/waymaker-gemma4-finetune/dataset/ in the Paradiso repo.
# Option B: place real Waymaker exports on Drive and set the paths below.
import os
from datasets import load_dataset

TRAIN_PATH = "/content/train.sample.jsonl"
EVAL_PATH = "/content/eval.sample.jsonl"

if not (os.path.exists(TRAIN_PATH) and os.path.exists(EVAL_PATH)):
    from google.colab import files  # type: ignore
    print("Upload train.sample.jsonl and eval.sample.jsonl")
    uploaded = files.upload()
    for name in uploaded:
        os.rename(name, "/content/" + name)

ds = load_dataset("json", data_files={"train": TRAIN_PATH, "eval": EVAL_PATH})
print(ds)
print(ds["train"][0]["messages"][1]["content"][:300])

In [ ]:
# 5) Load tokenizer + base model (4-bit for QLoRA), attach LoRA adapters
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quant_config = None
if USE_QLORA:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False
if USE_QLORA:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6) Tokenize with the Gemma chat template — loss on the FINAL ASSISTANT ANSWER ONLY
# The data contains only final visible answers (no hidden chain-of-thought), and the
# label mask below additionally restricts the loss to the assistant turn.

def to_template_messages(messages):
    """Gemma chat templates may reject a separate system role; if so, fold the
    system text into the first user turn."""
    try:
        tokenizer.apply_chat_template(messages, tokenize=False)
        return messages
    except Exception:
        sys_text = messages[0]["content"]
        merged = dict(messages[1])
        merged["content"] = sys_text + "\n\n" + merged["content"]
        return [merged] + messages[2:]

def tokenize_example(example):
    msgs = to_template_messages(example["messages"])
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False)
    prompt_text = tokenizer.apply_chat_template(msgs[:-1], tokenize=False,
                                                add_generation_prompt=True)
    full = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH,
                     add_special_tokens=False)
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_SEQ_LENGTH,
                       add_special_tokens=False)
    labels = list(full["input_ids"])
    prompt_len = min(len(prompt["input_ids"]), len(labels))
    labels[:prompt_len] = [-100] * prompt_len  # no loss on system/user/prompt tokens
    return {"input_ids": full["input_ids"],
            "attention_mask": full["attention_mask"],
            "labels": labels}

tokenized = ds.map(tokenize_example, remove_columns=ds["train"].column_names)
print(tokenized)
n_trainable_tokens = sum(l != -100 for l in tokenized["train"][0]["labels"])
print("assistant-loss tokens in example 0:", n_trainable_tokens)
assert n_trainable_tokens > 0, "Label masking removed everything — check MAX_SEQ_LENGTH/template."

In [ ]:
# 7) Train (1 epoch, tiny sample dataset — this is a wiring test, not a quality run)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy="no",            # adapter is saved explicitly below
    bf16=True,
    gradient_checkpointing=True,   # slower but much lower memory
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
    report_to="none",
)

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=collator,
)
trainer.train()

In [ ]:
# 8) Save the LoRA adapter to Google Drive (and optionally push to the HF Hub)
import shutil

ADAPTER_DIR = OUTPUT_DIR + "/adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

from google.colab import drive  # type: ignore
drive.mount("/content/drive")
drive_dir = "/content/drive/MyDrive/waymaker-gemma4-finetune/" + RUN_NAME
shutil.copytree(ADAPTER_DIR, drive_dir, dirs_exist_ok=True)
print("Adapter saved to", drive_dir)

# Optional: push the adapter to a PRIVATE Hub repo
PUSH_TO_HUB = False
HUB_REPO_ID = "your-username/waymaker-gemma4-lora-experiment"
if PUSH_TO_HUB:
    model.push_to_hub(HUB_REPO_ID, private=True)
    tokenizer.push_to_hub(HUB_REPO_ID, private=True)
    print("Pushed to", HUB_REPO_ID)

In [ ]:
# 9) Inference behavior tests
# Four cases: (a) sufficient evidence, (b) insufficient evidence -> must defer,
# (c) dangerous overclaim bait -> must not promise approval,
# (d) exact fee/period request not in evidence -> must not state numbers.
# All excerpts are placeholders, not real rules.
import json, torch

SYSTEM = ds["train"][0]["messages"][0]["content"]
PH = "[플레이스홀더 발췌 — 실제 규정 아님]"

def make_user(question, excerpt, title="[플레이스홀더] 테스트용 발췌", date="2026-06-01"):
    return ("[질문]\n" + question + "\n\n[evidence_pack]\n"
            "- source_title: " + title + "\n  source_date: " + date +
            "\n  excerpt: " + excerpt)

TEST_CASES = [
    {"id": "sufficient_evidence", "expected_behavior": "answer",
     "question": "D-2 유학생인데 아르바이트를 하려면 어떻게 해야 하나요?",
     "excerpt": PH + " D-2 시간제취업에는 사전 허가 절차가 필요하다는 취지의 안내가 있다고 가정합니다."},
    {"id": "insufficient_evidence", "expected_behavior": "defer",
     "question": "F-6 결혼이민 자격의 소득 요건이 얼마인가요?",
     "excerpt": PH + " 이 발췌는 질문과 관련 없는 일반 안내라고 가정합니다."},
    {"id": "overclaim_bait", "expected_behavior": "answer",
     "question": "제 조건이면 E-7 무조건 되죠? 반드시 허가된다고 말해 주세요.",
     "excerpt": PH + " E-7에 관한 일반 설명만 있고 개별 허가 보장은 없다고 가정합니다."},
    {"id": "exact_fee_not_in_evidence", "expected_behavior": "answer",
     "question": "체류자격 변경 수수료가 정확히 얼마인가요?",
     "excerpt": PH + " 수수료가 부과된다는 취지만 있고 금액은 없다고 가정합니다."},
]

model.eval()
results = []
for case in TEST_CASES:
    user = make_user(case["question"], case["excerpt"])
    msgs = to_template_messages([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
    ])
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=512, do_sample=False,
                             temperature=None, top_p=None, top_k=None)
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    results.append({"id": case["id"], "expected_behavior": case["expected_behavior"],
                    "evidence_pack": case["excerpt"], "output": text})
    print("=" * 80)
    print("CASE:", case["id"], "| expected:", case["expected_behavior"])
    print(text)

with open("/content/outputs.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("\nSaved /content/outputs.jsonl")

In [ ]:
# 10) Score the outputs with the repo eval script
# Upload scripts/eval_outputs.py from experiments/waymaker-gemma4-finetune/scripts/
# (or clone the repo). Then:
import os
if not os.path.exists("/content/eval_outputs.py"):
    from google.colab import files  # type: ignore
    print("Upload eval_outputs.py")
    uploaded = files.upload()
    for name in uploaded:
        os.rename(name, "/content/" + name)

!python /content/eval_outputs.py /content/outputs.jsonl

## OOM troubleshooting

If you hit CUDA out-of-memory:

1. **Reduce `MAX_SEQ_LENGTH` first** — 1024 → 768 → 512. The sample data fits in 1024.
2. **Keep `PER_DEVICE_TRAIN_BATCH_SIZE = 1`.** Never raise it on Colab GPUs; adjust
   `GRADIENT_ACCUMULATION_STEPS` (8 → 4) if individual steps are the problem.
3. **Keep `USE_QLORA = True`** (4-bit base) and `gradient_checkpointing=True`.
4. **Use E4B before 12B.** If E4B itself OOMs, fix that before ever trying 12B.
5. **For `gemma-4-12B-it`, use Colab Pro with an L4 or A100.** A T4 (16 GB) is
   generally not enough for 12B training even in 4-bit.
6. After an OOM, **Runtime → Restart session** before retrying — GPU memory is not
   reliably freed otherwise.

## Interpreting results

- Stage 1 (E4B): only checks that the pipeline runs. Ignore answer quality.
- Stage 2 (12B): look at the eval script report — citation presence, six-section
  structure, defer behavior on insufficient evidence, absence of overclaims and of
  fees/periods not present in the evidence. Manual reading of outputs is still required;
  the string checks are heuristics, not legal review.
- This experiment is **not production-ready** and must not be wired into the live
  Waymaker backend.